# Wan2GP on Google Colab

Sets up [Wan2GP](https://github.com/deepbeepmeep/Wan2GP) in a fresh GPU-backed Colab session.

Run the cells in order to prepare the runtime, install dependencies, and launch the Gradio interface. Click on the link in the output from the last cell to launch the app in your browser.

> **Colab VRAM note:** the free tier usually assigns a 15 GB T4 GPU. Most Wan2GP models exceed that budget; the Wan 2.2 TextImage2Video FastWan model works, producing roughly a 5 second 480p clip in about 8 minutes.


## 1. Confirm the accelerator

Choose `Runtime → Change runtime type` and select **GPU** before running anything else.

If this cell raises an error, go back to `Runtime → Change runtime type`, pick **GPU** and save.


In [1]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime → Change runtime type, select GPU (or TPU if GPUs are unavailable), save, then rerun this cell.'
    ) from exc


## 2. Configure the workspace path and optional persistent data storage


Set `USE_GOOGLE_DRIVE_DATA = True` if you want Google Drive to keep checkpoints, LoRAs, outputs and model caches across Colab restarts.


In [3]:
from pathlib import Path


# CHANGE THIS TO TRUE IF YOU WANT TO USE GOOGLE DRIVE FOR DATA STORAGE (PERSISTENT ACROSS SESSIONS)
USE_GOOGLE_DRIVE_DATA = True




DRIVE_MOUNT_POINT = Path('/content/drive')
WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
EPHEMERAL_DATA_ROOT = Path('/content/Wan2GP-data').resolve()
PERSISTENT_DATA_ROOT = (DRIVE_MOUNT_POINT / 'MyDrive' / 'Wan2GP-data').resolve()

if USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    WAN_DATA_ROOT = PERSISTENT_DATA_ROOT
    data_mode = 'Google Drive (persistent data)'
else:
    WAN_DATA_ROOT = EPHEMERAL_DATA_ROOT
    data_mode = 'Colab runtime disk (ephemeral data)'

WAN_CKPTS_DIR = (WAN_DATA_ROOT / 'ckpts').resolve()
WAN_LORAS_DIR = (WAN_DATA_ROOT / 'loras').resolve()
WAN_OUTPUTS_DIR = (WAN_DATA_ROOT / 'outputs').resolve()
WAN_CACHE_DIR = (WAN_DATA_ROOT / 'cache').resolve()
WAN_LTX2_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2').resolve()
WAN_LTX2_22B_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2_22B').resolve()

WAN2GP_ROOT.parent.mkdir(parents=True, exist_ok=True)
WAN_DATA_ROOT.mkdir(parents=True, exist_ok=True)
WAN_CKPTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_22B_LORAS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Wan2GP repository path: {WAN2GP_ROOT}')
print(f'Data storage mode: {data_mode}')
print(f'Data root: {WAN_DATA_ROOT}')
print(f'Checkpoints: {WAN_CKPTS_DIR}')
print(f'LoRAs: {WAN_LORAS_DIR}')
print(f'LTX-2 LoRAs: {WAN_LTX2_LORAS_DIR}')
print(f'LTX-2 22B LoRAs: {WAN_LTX2_22B_LORAS_DIR}')
print(f'Outputs: {WAN_OUTPUTS_DIR}')
print(f'Cache: {WAN_CACHE_DIR}')


Mounted at /content/drive
Wan2GP repository path: /content/Wan2GP
Data storage mode: Google Drive (persistent data)
Data root: /content/drive/MyDrive/Wan2GP-data
Checkpoints: /content/drive/MyDrive/Wan2GP-data/ckpts
LoRAs: /content/drive/MyDrive/Wan2GP-data/loras
LTX-2 LoRAs: /content/drive/MyDrive/Wan2GP-data/loras/ltx2
LTX-2 22B LoRAs: /content/drive/MyDrive/Wan2GP-data/loras/ltx2_22B
Outputs: /content/drive/MyDrive/Wan2GP-data/outputs
Cache: /content/drive/MyDrive/Wan2GP-data/cache


## 3. Download or update Wan2GP

Clone the repository if it is not present yet; otherwise pull the latest changes.


In [4]:
import shutil, subprocess
from pathlib import Path

def merge_directory_contents(source_dir: Path, destination_dir: Path) -> None:
    for child in list(source_dir.iterdir()):
        destination = destination_dir / child.name
        if destination.exists():
            if child.is_dir() and destination.is_dir():
                merge_directory_contents(child, destination)
                child.rmdir()
                continue
            raise RuntimeError(f'Cannot move {child} into {destination_dir}: {destination} already exists.')
        shutil.move(str(child), str(destination))

def attach_data_directory(repo_path: Path, data_path: Path) -> None:
    data_path.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        if repo_path.resolve() != data_path.resolve():
            raise RuntimeError(f'{repo_path} already points to {repo_path.resolve()}, expected {data_path}.')
        print(f'Using existing link: {repo_path} -> {data_path}')
        return

    if repo_path.exists():
        if not repo_path.is_dir():
            raise RuntimeError(f'Expected a directory at {repo_path}.')
        merge_directory_contents(repo_path, data_path)
        repo_path.rmdir()
    else:
        repo_path.parent.mkdir(parents=True, exist_ok=True)

    repo_path.symlink_to(data_path, target_is_directory=True)
    print(f'Linked {repo_path.name} -> {data_path}')

repo_url = 'https://github.com/deepbeepmeep/Wan2GP.git'
if WAN2GP_ROOT.exists():
    status = subprocess.run(
        ['git', '-C', str(WAN2GP_ROOT), 'status', '--porcelain'],
        check=True,
        capture_output=True,
        text=True,
    )
    if status.stdout.strip():
        print('Repository already exists and has local changes. Skipping git pull to preserve your persistent files.')
    else:
        print('Repository already exists. Pulling latest changes...')
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

attach_data_directory(WAN2GP_ROOT / 'ckpts', WAN_CKPTS_DIR)
attach_data_directory(WAN2GP_ROOT / 'loras', WAN_LORAS_DIR)
attach_data_directory(WAN2GP_ROOT / 'outputs', WAN_OUTPUTS_DIR)


Linked ckpts -> /content/drive/MyDrive/Wan2GP-data/ckpts
Linked loras -> /content/drive/MyDrive/Wan2GP-data/loras
Linked outputs -> /content/drive/MyDrive/Wan2GP-data/outputs


## 4. Install system dependencies

Install shared libraries needed for video and audio processing. If you see a warning about skipping an extra repository, it is safe to ignore.

In [5]:
import os, subprocess

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'

subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run([
    'sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=env)


CompletedProcess(args=['sudo', 'apt-get', 'install', '-y', '--no-install-recommends', 'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'], returncode=0)

## 5. Install Python dependencies

Install PyTorch, xformers and Wan2GP's Python packages.

In [8]:
import os, subprocess, sys

env = os.environ.copy()
env.setdefault('DEBIAN_FRONTEND', 'noninteractive')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'],
    check=True,
    env=env
)

# No reinstalar torch
# No desinstalar torch

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(WAN2GP_ROOT / 'requirements.txt')],
    check=True,
    env=env
)

print("Instalación terminada usando Torch ya instalado.")


Instalación terminada usando Torch ya instalado.


## 5b. Force a headless matplotlib backend

Ensure Wan2GP's preprocessing tools use the headless Agg backend so Step 6 launches cleanly in Colab.


In [9]:
from pathlib import Path

# Replace the TkAgg backend with the headless Agg backend if present.
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')


Replaced TkAgg with Agg in interact_tools.py.


## 6. Launch Wan2GP

Run the Gradio interface. You will find the gradio link in the output. Click on the link to access the UI. Keep the cell running to stay connected; stop it with the square **Stop** button when you are finished.

In [ ]:
import os, subprocess, sys, threading, time

env = os.environ.copy()
env['WAN_CACHE_DIR'] = str(WAN_CACHE_DIR)
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TRANSFORMERS_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'transformers')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')

cmd = [
    sys.executable,
    '-u',
    'wgp.py',
    '--listen',
    '--server-port', '7860',
    # '--share',  # Desactivado porque Gradio share está fallando
    '--profile', '5',
]

if USE_GOOGLE_DRIVE_DATA:
    print('Using Google Drive for checkpoints, LoRAs, outputs and caches.')
else:
    print('Using Colab runtime storage for checkpoints, LoRAs, outputs and caches.')

print('Launching Wan2GP…')

process = subprocess.Popen(
    cmd,
    cwd=str(WAN2GP_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

stop_event = threading.Event()

def keepalive():
    while not stop_event.is_set():
        time.sleep(45)
        if stop_event.is_set():
            break
        print('[keepalive] Notebook cell still running…')

keepalive_thread = threading.Thread(target=keepalive, daemon=True)
keepalive_thread.start()

print('Starting Cloudflare Tunnel…')

cloudflare = subprocess.Popen(
    [
        'bash',
        '-c',
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared && ./cloudflared tunnel --url http://127.0.0.1:7860'
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def cloudflare_output():
    for line in iter(cloudflare.stdout.readline, ''):
        print('[cloudflare]', line, end='')

cloudflare_thread = threading.Thread(target=cloudflare_output, daemon=True)
cloudflare_thread.start()

try:
    for line in iter(process.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Stopping Wan2GP…')
    process.terminate()
    cloudflare.terminate()
finally:
    stop_event.set()
    process.wait()
    try:
        cloudflare.terminate()
    except Exception:
        pass
    keepalive_thread.join(timeout=1)
    print(f'Wan2GP stopped (return code: {process.returncode}).')


Using Google Drive for checkpoints, LoRAs, outputs and caches.
Launching Wan2GP…
Starting Cloudflare Tunnel…
[cloudflare] 2026-06-15T14:38:32Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[cloudflare] 2026-06-15T14:38:32Z INF Requesting new quick Tunnel on trycloudflare.com...
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` in